<div class="alert alert-block alert-info">
<b>ℹ️ Note on Execution Order</b><br>
This notebook expects the preprocessed data to already be saved. If you haven't yet, please run <code>02-preprocessing.ipynb</code> first so the required data is available to load.
</div>

## Setup

Import libraries, initialize Ray, and start a Spark session on the distributed cluster.

In [1]:
!pip install -qq pyspark ray raydp

import os
import sys
import logging
from pathlib import Path

os.environ["RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO"] = "0"

import ray
import raydp

# Python-side loggers
logging.getLogger("ray").setLevel(logging.ERROR)
logging.getLogger("ray.data").setLevel(logging.ERROR)

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import pyspark.sql.functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    RegexTokenizer,
    Word2Vec,
    VectorAssembler,
    StandardScaler,
    StopWordsRemover,
    Imputer,
)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
# Clean notebook reruns
ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
    log_to_driver=False,
)

Python version:,3.11.6
Ray version:,2.54.0


In [3]:
spark = raydp.init_spark(
    app_name="FineWeb_Spark_on_Ray",
    num_executors=31,
    executor_cores=1,
    executor_memory="4GB",
)

## Data Loading

Load the preprocessed Parquet data and take a 20% sample for faster iteration.

In [4]:
from fineweb_spark.paths import INTERIM_DATA_DIR

# Load preprocessed data
df = spark.read.parquet(str(INTERIM_DATA_DIR / "fineweb_preprocessed.parquet")) \
          .select("text", "token_count", "label") \
          .sample(fraction=0.2, seed=42)

## Data Exploration

Inspect row count, schema, estimated size, and a few sample rows.

In [5]:
print("===== DATASET INFO =====")

# number of rows
row_count = df.count()
print(f"Rows: {row_count:,}")

# number of columns
col_count = len(df.columns)
print(f"Columns: {col_count}")

# schema
print("\nSchema:")
df.printSchema()

# Estimated dataset size:
sample_rows = 1000
sample_pdf = df.limit(sample_rows).toPandas()

row_size_bytes = sample_pdf.memory_usage(deep=True).sum() / sample_rows
estimated_size_gb = (row_size_bytes * row_count) / (1024**3)

print("===== DATASET SIZE =====")
print(f"Estimated dataset size: {estimated_size_gb:.2f} GB")

print("\n===== DATASET EXAMPLES =====")
df.show(10, truncate=80)

===== DATASET INFO =====
Rows: 142,415
Columns: 3

Schema:
root
 |-- text: string (nullable = true)
 |-- token_count: long (nullable = true)
 |-- label: integer (nullable = true)

===== DATASET SIZE =====
Estimated dataset size: 1.19 GB

===== DATASET EXAMPLES =====
+--------------------------------------------------------------------------------+-----------+-----+
|                                                                            text|token_count|label|
+--------------------------------------------------------------------------------+-----------+-----+
|Gravitational Wave Astronomy Will Be The Next Great Scientific Frontier\nBy E...|        730|    4|
|Locked inside the littlest objects of the solar system—asteroids, comets, and...|        448|    5|
|This article explains the history and theory of the Socratic method of teachi...|       1782|    4|
|Most everyone can identify a maple leaf, oak leaf, white birch bark, and a pi...|        592|    4|
|Copyright © 2007 Dorling 

## Train / Val / Test Split

Split the data 60 / 20 / 20 for training, validation, and testing.

In [6]:
# # Split into Train / Validation / Test (60 / 20 / 20)
train_data, val_data, test_data = df.randomSplit([0.6, 0.2, 0.2], seed=42)

print(f"Train: {train_data.count()}  Val: {val_data.count()}  Test: {test_data.count()}")

Train: 85524  Val: 28398  Test: 28493


## Feature Engineering

Build a pipeline that tokenizes text, removes stop words, trains Word2Vec embeddings, imputes and scales `token_count`, then assembles everything into a single feature vector.

In [7]:
# Split raw text into lowercase word tokens, removing punctuation
tokenizer = RegexTokenizer(inputCol="text", outputCol="raw_words", pattern="\\W+", toLowercase=True)

# Remove common English stop words to reduce noise and vocabulary size.
remover = StopWordsRemover(inputCol="raw_words", outputCol="filtered_words")

# Map filtered words to 10-dimensional dense vectors. Words appearing fewer than 500 times are ignored.
word2vec = Word2Vec(vectorSize=10, minCount=500, inputCol="filtered_words", outputCol="text_features")

# Impute missing token_count values using the mean strategy, creating a new column token_count_imputed
imputer = Imputer(inputCols=["token_count"], outputCols=["token_count_imputed"], strategy="mean")

# Assemble the imputed token_count into a vector for scaling
vec_assembler_num = VectorAssembler(inputCols=["token_count_imputed"], outputCol="token_vec")
scaler = StandardScaler(inputCol="token_vec", outputCol="token_scaled", withStd=True, withMean=True)

# Combine text features and scaled token_count into a single feature vector for modeling
final_assembler = VectorAssembler(
    inputCols=["text_features", "token_scaled"], 
    outputCol="features"
)

In [8]:
feature_pipeline = Pipeline(stages=[
    tokenizer,
    remover,
    word2vec,
    imputer,
    vec_assembler_num,
    scaler,
    final_assembler
])

In [9]:
%%time
print("Fitting preprocessing pipeline...")
feature_model = feature_pipeline.fit(train_data)
print("Preprocessing pipeline fit complete.")

Fitting preprocessing pipeline...
Preprocessing pipeline fit complete.
CPU times: user 494 ms, sys: 199 ms, total: 693 ms
Wall time: 9min 43s


In [10]:
# Transform all splits and cache prepared datasets
print("Transforming and caching datasets...")

train_prepared = feature_model.transform(train_data).cache()
val_prepared   = feature_model.transform(val_data).cache()
test_prepared  = feature_model.transform(test_data).cache()

# materialize cache
train_prepared.count()
val_prepared.count()
test_prepared.count()

print("Cached prepared split sizes:")
print(f"Train prepared: {train_prepared.count():,}")
print(f"Val prepared  : {val_prepared.count():,}")
print(f"Test prepared : {test_prepared.count():,}")

Transforming and caching datasets...
Cached prepared split sizes:
Train prepared: 85,524
Val prepared  : 28,389
Test prepared : 28,489


In [11]:
# inspect columns
train_prepared.select("features", "label").show(5, truncate=85)

+-------------------------------------------------------------------------------------+-----+
|                                                                             features|label|
+-------------------------------------------------------------------------------------+-----+
|[0.1891847732936126,-0.12749595097009117,0.16511493811214514,0.2468289164357589,-0...|    4|
|[0.1252200756039116,0.08781548857402703,0.14249290595700012,0.05542941399208263,-0...|    3|
|[0.10219370454353273,0.031701255061048504,0.17871357213680147,0.09514247719465474,...|    3|
|[0.12744946863390824,-0.21064392626524356,0.015052843092678068,0.04142107678633024...|    3|
|[0.222718165655508,-0.17424884808283894,-0.05060864438656744,0.17795249991367795,-...|    3|
+-------------------------------------------------------------------------------------+-----+
only showing top 5 rows



## Model Training

Train two Random Forest classifiers with different hyperparameters and compare their performance.

In [12]:
# Define model 1
rf1 = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=5,
    maxDepth=2
)

In [13]:
%%time
print("Training Model 1...")
model1 = rf1.fit(train_prepared)
print("Model 1 complete.")

Training Model 1...
Model 1 complete.
CPU times: user 0 ns, sys: 14.9 ms, total: 14.9 ms
Wall time: 3.68 s


In [14]:
# Define model 2
rf2 = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=20,
    maxDepth=8
)

In [15]:
%%time
print("Training Model 2...")
model2 = rf2.fit(train_prepared)
print("Model 2 complete.")

Training Model 2...
Model 2 complete.
CPU times: user 0 ns, sys: 12.7 s, total: 12.7 s
Wall time: 1min 11s


## Evaluation

Compute accuracy on train, validation, and test splits for both models, then select the best one.

In [16]:
# Evaluate both models
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

# Model 1 predictions
train_preds1 = model1.transform(train_prepared)
val_preds1   = model1.transform(val_prepared)
test_preds1  = model1.transform(test_prepared)

train_acc1 = evaluator.evaluate(train_preds1)
val_acc1   = evaluator.evaluate(val_preds1)
test_acc1  = evaluator.evaluate(test_preds1)

print("\nModel 1 — RF (numTrees=5, maxDepth=2)")
print(f" Train Accuracy: {train_acc1:.4f}")
print(f" Val Accuracy  : {val_acc1:.4f}")
print(f" Test Accuracy : {test_acc1:.4f}")


# Model 2 predictions
train_preds2 = model2.transform(train_prepared)
val_preds2   = model2.transform(val_prepared)
test_preds2  = model2.transform(test_prepared)

train_acc2 = evaluator.evaluate(train_preds2)
val_acc2   = evaluator.evaluate(val_preds2)
test_acc2  = evaluator.evaluate(test_preds2)

print("\nModel 2 — RF (numTrees=20, maxDepth=8)")
print(f" Train Accuracy: {train_acc2:.4f}")
print(f" Val Accuracy  : {val_acc2:.4f}")
print(f" Test Accuracy : {test_acc2:.4f}")


Model 1 — RF (numTrees=5, maxDepth=2)
 Train Accuracy: 0.6133
 Val Accuracy  : 0.6089
 Test Accuracy : 0.6083

Model 2 — RF (numTrees=20, maxDepth=8)
 Train Accuracy: 0.7009
 Val Accuracy  : 0.6865
 Test Accuracy : 0.6866


In [17]:
# Compare results
print("\n--- Comparison ---")
print(f"{'Model':<35} {'Train':>8} {'Val':>8} {'Test':>8}")
print(f"{'RF (numTrees=5, maxDepth=2)':<35} {train_acc1:>8.4f} {val_acc1:>8.4f} {test_acc1:>8.4f}")
print(f"{'RF (numTrees=20, maxDepth=8)':<35} {train_acc2:>8.4f} {val_acc2:>8.4f} {test_acc2:>8.4f}")


--- Comparison ---
Model                                  Train      Val     Test
RF (numTrees=5, maxDepth=2)           0.6133   0.6089   0.6083
RF (numTrees=20, maxDepth=8)          0.7009   0.6865   0.6866


In [18]:
# Pick best model
results = [
    ("RF (numTrees=5, maxDepth=2)", model1, test_acc1),
    ("RF (numTrees=20, maxDepth=8)", model2, test_acc2),
]

best_name, best_model, best_acc = max(results, key=lambda x: x[2])

print("\nBest model:")
print(f" Name          : {best_name}")
print(f" Test accuracy : {best_acc:.4f}")


Best model:
 Name          : RF (numTrees=20, maxDepth=8)
 Test accuracy : 0.6866


## Save & Cleanup

Persist the best model to disk, then release cached DataFrames and shut down Ray.

In [19]:
# Save model
from fineweb_spark.paths import MODELS_DIR

best_model.write().overwrite().save(str(MODELS_DIR / "best_model1"))

print(f"Model saved")

Model saved


In [20]:
# Cleanup cache and shutdown the ray
train_prepared.unpersist()
val_prepared.unpersist()
test_prepared.unpersist()

ray.shutdown()
print("\nDone.")


Done.


## Fitting Analysis

The dataset contains **142,415 rows** after 20% random sampling (3 classes: `int_score` 3, 4, and 5).

| Model | Train | Val | Test | Train–Test Gap |
|---|---|---|---|---|
| RF (numTrees=5, maxDepth=2) | 0.6133 | 0.6089 | 0.6083 | 0.0050 |
| RF (numTrees=20, maxDepth=8) | 0.7009 | 0.6865 | 0.6866 | 0.0143 |

**1. Where does the model fit on the underfitting / overfitting spectrum?**
- **Model 1** (shallow, few trees) sits toward **underfitting** — the 0.5 pp train–test gap shows no overfitting, but maxDepth=2 limits the model to very simple decision boundaries, leaving significant accuracy on the table.
- **Model 2** (deeper, more trees) shows **mild overfitting** (1.4 pp gap) but is still well-generalized. It sits in a better position on the bias–variance tradeoff.

**2. Which model performs best and why?**
**Model 2** (RF `numTrees=20, maxDepth=8`) — **68.66% test accuracy** (+6.8 pp over Model 1). Deeper trees can capture higher-order interactions between Word2Vec embeddings and token count, and 20 trees provide stronger ensemble variance reduction than 5. The accuracy gain is real generalization, not memorization, since the train–test gap remains small.

**3. Next models planned for Milestone 4 and why?**
- **Gradient Boosted Trees (GBTClassifier)**: Sequential boosting iteratively corrects residual errors — better suited for the nuanced boundaries between quality scores 3, 4, and 5.
- **XGBoost (SparkXGBClassifier)**: Built-in L1/L2 regularization should reduce the mild overfitting seen in Model 2 while maintaining high accuracy.
- **Feature Engineering**: Adding TF-IDF, sentence count, average word length, and punctuation density captures stylistic signals beyond token-level semantics that likely correlate with educational quality.


## Conclusion

**1. What is the conclusion of the 1st model?**
Model 1 (RF `numTrees=5`, `maxDepth=2`) achieves **60.83% test accuracy** on a 3-class quality classification task (`int_score` 3, 4, and 5). The negligible train–test gap (0.5 pp) confirms the model generalizes well but is capacity-constrained — shallow trees cannot capture the complex linguistic patterns that distinguish document quality. The pipeline is validated (Word2Vec embeddings + token count carry real signal), but the model is too simple to exploit it fully.

**2. What can be done to possibly improve it?**
- **Increase model capacity**: Model 2 (`numTrees=20`, `maxDepth=8`) already demonstrates this — test accuracy jumps to 68.66% with only mild overfitting (1.4 pp gap).
- **Richer features**: Higher-dimensional Word2Vec, TF-IDF, sentence count, average word length, and punctuation density would better represent educational quality.
- **More training data**: Only 20% of the corpus was used (~85.5K training examples); scaling up reduces variance.
- **Boosted models**: GBTClassifier or XGBoost should close the remaining accuracy gap with built-in regularization.
- **True 3-class modeling**: Ensuring all three label classes (3, 4, 5) are properly handled in preprocessing and evaluation will give a more accurate picture of model performance.

**3. How did distributed computing help with this task?**
The FineWeb-Edu dataset is too large to process on a single machine. Running PySpark on 31 Ray executor cores enabled:
- **Parallel data loading and preprocessing** across partitions, handling ~712K rows in the full dataset (142K after 20% sampling).
- **Distributed Word2Vec training** using partitioned gradient updates — completing in minutes instead of hours.
- **Parallel Random Forest construction** — Model 1 trained in ~4 s and Model 2 in ~71 s on 31 cores; local training would be infeasible at this scale.
- **Scalable inference** — `.transform()` evaluated all three splits (~142K rows total) in seconds by processing partitions concurrently.
